# FlexDC raw inference + end-to-end validation v4

Raw-model notebook for the new W2-dense dataset. It keeps the original Colab workflow: install deps, clone repos, define paths, optional Drive copy, W&B login, predict, optimize, end-to-end FlexDC validation, compact tables, and constants playground.

Default behavior is **general/broad**: no local P/R bounds and `OBJECTIVE_WEIGHTS="auto"` (equal weights for FlexDC). Turn on focused bounds only when intentionally testing the W2 local feasible band.

## 1. Environment controls

In [ ]:
from pathlib import Path
import os

RUN_ENV = "colab"  # "colab" or "local"
WORKSPACE = Path("/content/workspace") if RUN_ENV == "colab" else Path.cwd().parent.parent

COMDER_REPO_URL = "https://github.com/NetherMoon/CONDOR-FLEXDC"
FLEXDC_REPO_URL = "https://github.com/amenon871/FlexDC"
COMDER_BRANCH = "main"
FLEXDC_BRANCH = "main"

DEVICE = "cuda"  # "cuda", "cpu", or "auto"

WANDB_ENTITY = "amenon06-boston-university"
WANDB_PROJECT = "flexdc-unified-inference"
WANDB_MODE = "online"  # "online", "offline", or "disabled"

print("RUN_ENV:", RUN_ENV)
print("WORKSPACE:", WORKSPACE)

## 2. Install dependencies

In [ ]:
%pip install -q wandb pandas numpy scipy scikit-learn tqdm matplotlib tabulate openpyxl

## 3. Clone repositories — run in Colab

In [ ]:
if RUN_ENV == "colab":
    !rm -rf "$WORKSPACE"
    !mkdir -p "$WORKSPACE"
    !git clone --branch "$COMDER_BRANCH" "$COMDER_REPO_URL" "$WORKSPACE/comder-main"
    !git clone --branch "$FLEXDC_BRANCH" "$FLEXDC_REPO_URL" "$WORKSPACE/flexdc-sim"
else:
    print("Skipping clone for local run.")

## 4. Define paths

In [ ]:
import sys

COMDER_ROOT = WORKSPACE / "comder-main"
FLEXDC_ROOT = WORKSPACE / "flexdc-sim"
AM_FLEXDC_ROOT = COMDER_ROOT / "am_flexdc"
TRAIN_DIR = AM_FLEXDC_ROOT / "train"
MODELS_DIR = AM_FLEXDC_ROOT / "models"
RESULTS_DIR = AM_FLEXDC_ROOT / "results" / "unified_eval_runs"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_TAG = "newqos_plus_w2dense_v1"
PILOT_DIR = AM_FLEXDC_ROOT / "data" / "pilots" / "traditionaliso_newqos_pilot_plus_w2_dense_v1_flexdc_configured_objective"
RESULTS_CSV = PILOT_DIR / "traditional_iso16_newqos_plus_w2dense_AQA_combined_grid_search_results.csv"
DIAGNOSTICS_CSV = PILOT_DIR / "traditional_iso16_newqos_plus_w2dense_AQA_combined_grid_search_diagnostics.csv"

if str(TRAIN_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_DIR))

print("COMDER_ROOT:", COMDER_ROOT)
print("FLEXDC_ROOT:", FLEXDC_ROOT)
print("TRAIN_DIR:", TRAIN_DIR)
print("PILOT_DIR:", PILOT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

## 5. Optional Google Drive copy — run only if data/model/scripts are not already in GitHub

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
#
# PILOT_DIR.mkdir(parents=True, exist_ok=True)
# !cp "/content/drive/MyDrive/path/traditional_iso16_newqos_plus_w2dense_AQA_combined_grid_search_results.csv" "$PILOT_DIR/"
# !cp "/content/drive/MyDrive/path/traditional_iso16_newqos_plus_w2dense_AQA_combined_grid_search_diagnostics.csv" "$PILOT_DIR/"
#
# (MODELS_DIR / "flexdc_raw").mkdir(parents=True, exist_ok=True)
# !cp "/content/drive/MyDrive/path/am_flexdc_raw_newqos_plus_w2dense_v1_wandb_v2_state_dict.pt" "$MODELS_DIR/flexdc_raw/"
#
# # If v4 scripts are not committed yet, copy them too:
# !cp "/content/drive/MyDrive/path/am_unified_optimize_one_v2.py" "$TRAIN_DIR/"
# !cp "/content/drive/MyDrive/path/am_unified_end_to_end_eval_raw_report_v4.py" "$TRAIN_DIR/"

## 6. Required-file check

In [ ]:
required = [
    TRAIN_DIR / "data_center_model.py",
    TRAIN_DIR / "am_unified_training_utilities.py",
    TRAIN_DIR / "am_unified_predict_one.py",
    TRAIN_DIR / "am_unified_optimize_one_v2.py",
    TRAIN_DIR / "am_unified_end_to_end_eval_raw_report_v4.py",
    RESULTS_CSV,
    DIAGNOSTICS_CSV,
    FLEXDC_ROOT / "src" / "peacsim" / "am_data_extraction_wizard.py",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    print("Missing required files:")
    for p in missing:
        print(" -", p)
    raise FileNotFoundError("Fix missing files before continuing.")
print("All required files found.")

## 7. W&B login

In [ ]:
if WANDB_MODE != "disabled":
    import os, getpass, wandb
    os.environ["WANDB_MODE"] = WANDB_MODE
    os.environ.pop("WANDB_BASE_URL", None)
    ok = False
    if RUN_ENV == "colab":
        try:
            from google.colab import userdata
            key = userdata.get("WANDB_API_KEY")
        except Exception:
            key = None
        if key:
            os.environ["WANDB_API_KEY"] = key
            ok = wandb.login(key=key, relogin=True, verify=True)
    if not ok:
        try:
            ok = wandb.login(relogin=True, verify=True)
        except Exception as exc:
            print("W&B login did not complete automatically:", repr(exc))
            key = getpass.getpass("Paste W&B API key: ")
            os.environ["WANDB_API_KEY"] = key
            ok = wandb.login(key=key, relogin=True, verify=True)
    if not ok:
        raise RuntimeError("W&B login failed. Set WANDB_MODE='disabled' to run without W&B.")
    print("W&B login verified.")
else:
    print("W&B disabled.")

## 8. Model and target configuration

In [ ]:
TARGET_FAMILY = "flexdc"
TARGET_MODE = "raw"
RAW_QOS_AGGREGATION = "mean"
USE_NORM_COST = "auto"   # FlexDC resolves to False; CONDOR resolves to True.
USE_NORM_PR = "true"     # Must match training.

MODEL_FILE = MODELS_DIR / "flexdc_raw" / "am_flexdc_raw_newqos_plus_w2dense_v1_wandb_v2_state_dict.pt"

if not MODEL_FILE.exists():
    raise FileNotFoundError(f"Model checkpoint not found: {MODEL_FILE}")

print("Model variant:", TARGET_FAMILY, TARGET_MODE)
print("Model file:", MODEL_FILE)

## 9. Scenario controls — edit this cell for a custom config

In [ ]:
# Presets: "W2_LU", "W2_HU", or "CUSTOM".
SCENARIO = "W2_LU"

SERVER_COUNT = 1000
EXPERIMENT_CONFIG = FLEXDC_ROOT / "configs" / "experiment" / "new_iso" / "traditional_signal" / "generated_server_counts" / "exp_traditional_iso16_servers_1000.ini"

if SCENARIO == "W2_LU":
    WORKLOAD_CONFIG = FLEXDC_ROOT / "configs" / "workload" / "W2-short-qos5_4.5_4_3.5.ini"
    UTILIZATION = 0.60
    START_PBAR = 0.472128
    START_R = 0.102206
    START_WEIGHTS = "0.258019617529,0.250894690209,0.252694830182,0.23839086208"
    FOCUSED_PBAR_MIN, FOCUSED_PBAR_MAX, FOCUSED_R_MAX = 0.464, 0.480, 0.140
elif SCENARIO == "W2_HU":
    WORKLOAD_CONFIG = FLEXDC_ROOT / "configs" / "workload" / "W2-short-qos5555.ini"
    UTILIZATION = 0.80
    START_PBAR = 0.576789
    START_R = 0.138748
    START_WEIGHTS = "0.25881071977840475,0.2545059365273737,0.24568691042720164,0.24099643326701997"
    FOCUSED_PBAR_MIN, FOCUSED_PBAR_MAX, FOCUSED_R_MAX = 0.570, 0.593, 0.180
elif SCENARIO == "CUSTOM":
    WORKLOAD_CONFIG = FLEXDC_ROOT / "configs" / "workload" / "W2-short-qos5_4.5_4_3.5.ini"
    UTILIZATION = 0.60
    START_PBAR = 0.472128
    START_R = 0.102206
    START_WEIGHTS = "0.25,0.25,0.25,0.25"
    FOCUSED_PBAR_MIN, FOCUSED_PBAR_MAX, FOCUSED_R_MAX = None, None, None
else:
    raise ValueError(f"Unknown SCENARIO: {SCENARIO}")

ITERATIONS = 2000
LR = 0.0025

# IMPORTANT: Leave auto by default. For FlexDC raw, auto means equal component weights [1,1,1].
# This is the neutral baseline. Only change this for experiments.
OBJECTIVE_WEIGHTS = "auto"

# Generalization mode: default False. If False, optimizer uses normal broad workload-derived bounds.
# Set True only to intentionally focus the search around the local W2 feasible region.
USE_FOCUSED_BOUNDS = False

print("Scenario:", SCENARIO)
print("Workload config:", WORKLOAD_CONFIG)
print("Utilization:", UTILIZATION)
print("Start P/R/W:", START_PBAR, START_R, START_WEIGHTS)
print("Objective weights:", OBJECTIVE_WEIGHTS, "(auto = equal [1,1,1] for FlexDC)")
print("Use focused bounds:", USE_FOCUSED_BOUNDS)

## 10. Paper-form constants playground for reporting only

In [ ]:
# These constants are used only to reconstruct/report the paper-form objective from raw validation outputs.
# They are NOT the raw-model optimization weights.
REPORT_CTRACK_PSI = 1.0
REPORT_CTRACK_MU = 10.0
REPORT_CTRACK_GAMMA = 0.3
REPORT_QOS_BETA = 20.0
REPORT_QOS_RHO = 2.0
REPORT_QOS_THRESHOLD = 0.1

print({
    "psi": REPORT_CTRACK_PSI,
    "mu": REPORT_CTRACK_MU,
    "gamma": REPORT_CTRACK_GAMMA,
    "beta": REPORT_QOS_BETA,
    "rho": REPORT_QOS_RHO,
    "delta": REPORT_QOS_THRESHOLD,
})

## 11. Helper for running commands

In [ ]:
import subprocess, sys, shlex
from pathlib import Path

def run_cmd(cmd, cwd=None):
    print("Running:")
    print(" ".join(shlex.quote(str(x)) for x in cmd))
    subprocess.run([str(x) for x in cmd], cwd=str(cwd) if cwd else None, check=True)

def append_optional_bounds(cmd):
    if USE_FOCUSED_BOUNDS:
        if FOCUSED_PBAR_MIN is not None:
            cmd += ["--pbar-min-kw-per-server", str(FOCUSED_PBAR_MIN)]
        if FOCUSED_PBAR_MAX is not None:
            cmd += ["--pbar-max-kw-per-server", str(FOCUSED_PBAR_MAX)]
        if FOCUSED_R_MAX is not None:
            cmd += ["--r-max-kw-per-server", str(FOCUSED_R_MAX)]
    return cmd

def common_model_args():
    return [
        "--model-file", MODEL_FILE,
        "--norm-source-results-csv", RESULTS_CSV,
        "--workload-config", WORKLOAD_CONFIG,
        "--experiment-config", EXPERIMENT_CONFIG,
        "--target-family", TARGET_FAMILY,
        "--target-mode", TARGET_MODE,
        "--raw-qos-aggregation", RAW_QOS_AGGREGATION,
        "--use-norm-cost", USE_NORM_COST,
        "--use-norm-pr", USE_NORM_PR,
        "--server-count", SERVER_COUNT,
        "--utilization", UTILIZATION,
        "--objective-weights", OBJECTIVE_WEIGHTS,
        "--device", DEVICE,
    ]

## 12. Compile scripts

In [ ]:
run_cmd([
    sys.executable, "-m", "py_compile",
    "am_unified_predict_one.py",
    "am_unified_optimize_one_v2.py",
    "am_unified_end_to_end_eval_raw_report_v4.py",
], cwd=TRAIN_DIR)

## 13. Predict one starting configuration

In [ ]:
PREDICT_OUT = RESULTS_DIR / f"predict_one_{TARGET_FAMILY}_{TARGET_MODE}_{SCENARIO}.json"
cmd = [sys.executable, "am_unified_predict_one.py"] + common_model_args() + [
    "--pbar-kw-per-server", START_PBAR,
    "--r-kw-per-server", START_R,
    "--weights", START_WEIGHTS,
    "--out-json", PREDICT_OUT,
    "--wandb-project", WANDB_PROJECT,
    "--wandb-entity", WANDB_ENTITY,
    "--wandb-run-name", f"predict-one-raw-{SCENARIO}",
    "--wandb-mode", WANDB_MODE,
]
run_cmd(cmd, cwd=TRAIN_DIR)
print("Saved:", PREDICT_OUT)

## 14. Optimize only

In [ ]:
OPT_DIR = RESULTS_DIR / f"optimize_only_{TARGET_FAMILY}_{TARGET_MODE}_{SCENARIO}_v4"
cmd = [sys.executable, "am_unified_optimize_one_v2.py"] + common_model_args() + [
    "--start-pbar-kw-per-server", START_PBAR,
    "--start-r-kw-per-server", START_R,
    "--start-weights", START_WEIGHTS,
    "--iterations", ITERATIONS,
    "--lr", LR,
    "--out-dir", OPT_DIR,
    "--wandb-project", WANDB_PROJECT,
    "--wandb-entity", WANDB_ENTITY,
    "--wandb-run-name", f"optimize-one-raw-{SCENARIO}-v4",
    "--wandb-mode", WANDB_MODE,
]
cmd = append_optional_bounds(cmd)
run_cmd(cmd, cwd=TRAIN_DIR)
print("Optimization folder:", OPT_DIR)

## 15. End-to-end: optimize then validate in FlexDC

In [ ]:
E2E_DIR = RESULTS_DIR / f"e2e_{TARGET_FAMILY}_{TARGET_MODE}_{SCENARIO}_N{SERVER_COUNT}_U{int(UTILIZATION*100):03d}_v4"
cmd = [sys.executable, "am_unified_end_to_end_eval_raw_report_v4.py"] + common_model_args() + [
    "--start-pbar-kw-per-server", START_PBAR,
    "--start-r-kw-per-server", START_R,
    "--start-weights", START_WEIGHTS,
    "--iterations", ITERATIONS,
    "--lr", LR,
    "--flexdc-root", FLEXDC_ROOT,
    "--flexdc-python", sys.executable,
    "--run-flexdc",
    "--out-dir", E2E_DIR,
    "--report-ctrack-psi", REPORT_CTRACK_PSI,
    "--report-ctrack-mu", REPORT_CTRACK_MU,
    "--report-ctrack-gamma", REPORT_CTRACK_GAMMA,
    "--report-qos-beta", REPORT_QOS_BETA,
    "--report-qos-rho", REPORT_QOS_RHO,
    "--report-qos-threshold", REPORT_QOS_THRESHOLD,
    "--wandb-project", WANDB_PROJECT,
    "--wandb-entity", WANDB_ENTITY,
    "--wandb-run-name", f"e2e-raw-{SCENARIO}-v4",
    "--wandb-mode", WANDB_MODE,
]
cmd = append_optional_bounds(cmd)
run_cmd(cmd, cwd=TRAIN_DIR)
print("Evaluation folder:", E2E_DIR)

## 16. Compact raw predicted-vs-actual table

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

if "E2E_DIR" not in globals():
    raise NameError("Run the end-to-end cell first.")

key_path = Path(E2E_DIR) / "end_to_end_raw_key_summary.csv"
summary_path = Path(E2E_DIR) / "end_to_end_validation_summary.csv"
if key_path.exists():
    df = pd.read_csv(key_path)
elif summary_path.exists():
    df = pd.read_csv(summary_path)
else:
    raise FileNotFoundError(f"Missing E2E summary in {E2E_DIR}")

cols = [
    "Configuration",
    "Pbar_kw_per_server", "R_kw_per_server", "Pbar_plus_R", "Pbar_minus_R", "Weights",
    "Predicted_flexdc_M_RSR", "Actual_flexdc_M_RSR",
    "Predicted_raw_Ctrack_Epsilon_90th", "Actual_raw_Ctrack_Epsilon_90th",
    "Predicted_raw_qos_probability_mean", "Actual_raw_qos_probability_mean",
    "QoS_Violation_Ratio", "Max_QoS_Delay_Probability", "Mean_QoS_Delay_Probability",
    "Tracking_Pass", "QoS_Pass_CurrentLogic", "Both_Pass_CurrentLogic",
    "Predicted_PaperObjective_Approx", "paper_objective_from_raw",
]
cols = [c for c in cols if c in df.columns]
view = df[cols].copy()
for c in view.columns:
    if c not in {"Configuration", "Weights"} and pd.api.types.is_numeric_dtype(view[c]):
        view[c] = view[c].astype(float).round(6)

display(Markdown("### Raw FlexDC predicted vs actual key table"))
display(view)
print("Source:", key_path if key_path.exists() else summary_path)

## 17. Constants playground on saved validation outputs

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

# Edit these and rerun this cell. These affect only reporting, not the validated FlexDC outputs.
PLAY_PSI = REPORT_CTRACK_PSI
PLAY_MU = REPORT_CTRACK_MU
PLAY_GAMMA = REPORT_CTRACK_GAMMA
PLAY_BETA = REPORT_QOS_BETA
PLAY_RHO = REPORT_QOS_RHO
PLAY_DELTA = REPORT_QOS_THRESHOLD

def softplus(x):
    x = np.asarray(x, dtype=float)
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0.0)

def parse_probs(x):
    if pd.isna(x) or str(x).strip() == "":
        return []
    return [float(v) for v in json.loads(str(x))]

path = Path(E2E_DIR) / "end_to_end_validation_summary.csv"
df = pd.read_csv(path)
rows = []
for _, r in df.iterrows():
    probs = parse_probs(r.get("QoS_Delay_Probabilities", ""))
    if probs:
        actual_ctrack = float(PLAY_PSI * softplus(PLAY_MU * (float(r["Ctrack_Epsilon_90th"]) - PLAY_GAMMA)))
        actual_cqos = float(PLAY_BETA * np.sum(softplus(PLAY_RHO * (np.asarray(probs) - PLAY_DELTA))))
        actual_obj = float(r["Simulator_RSR_Total_Cost"]) + actual_ctrack + actual_cqos
    else:
        actual_ctrack = actual_cqos = actual_obj = np.nan

    pred_obj = pred_ctrack = pred_cqos = np.nan
    if {"Predicted_flexdc_M_RSR", "Predicted_raw_Ctrack_Epsilon_90th", "Predicted_raw_qos_probability_mean"}.issubset(r.index):
        q_mean = float(r["Predicted_raw_qos_probability_mean"])
        job_count = max(len(probs), 4)
        pred_ctrack = float(PLAY_PSI * softplus(PLAY_MU * (float(r["Predicted_raw_Ctrack_Epsilon_90th"]) - PLAY_GAMMA)))
        pred_cqos = float(PLAY_BETA * job_count * softplus(PLAY_RHO * (q_mean - PLAY_DELTA)))
        pred_obj = float(r["Predicted_flexdc_M_RSR"]) + pred_ctrack + pred_cqos

    rows.append({
        "Configuration": r["Configuration"],
        "Pbar": r["Pbar_kw_per_server"],
        "R": r["R_kw_per_server"],
        "Actual_M_RSR": r.get("Simulator_RSR_Total_Cost", np.nan),
        "Actual_Ctrack_const": actual_ctrack,
        "Actual_CQoS_const": actual_cqos,
        "Actual_PaperObjective_const": actual_obj,
        "Pred_Ctrack_approx": pred_ctrack,
        "Pred_CQoS_approx": pred_cqos,
        "Pred_PaperObjective_approx": pred_obj,
        "QoS_probs": json.dumps(probs),
    })

out = pd.DataFrame(rows)
for c in out.columns:
    if c not in {"Configuration", "QoS_probs"} and pd.api.types.is_numeric_dtype(out[c]):
        out[c] = out[c].astype(float).round(6)

display(Markdown(f"### Paper-form objective with psi={PLAY_PSI}, mu={PLAY_MU}, gamma={PLAY_GAMMA}, beta={PLAY_BETA}, rho={PLAY_RHO}, delta={PLAY_DELTA}"))
display(out)

# Note: predicted CQoS is approximate because current raw model predicts only QoS mean, not per-job-type Pj.

## 18. Inspect trajectory

In [ ]:
traj_path = Path(E2E_DIR) / "optimization_trajectory.csv"
if traj_path.exists():
    traj = pd.read_csv(traj_path)
    display(traj.tail(10))
else:
    print("No trajectory found yet:", traj_path)

## 19. Download outputs

In [ ]:
from google.colab import files
from pathlib import Path

if 'E2E_DIR' in globals() and Path(E2E_DIR).exists():
    zip_path = Path('/content') / f"{Path(E2E_DIR).name}.zip"
    !zip -r "{zip_path}" "{E2E_DIR}"
    files.download(str(zip_path))
else:
    print("No E2E_DIR found yet. Run the full end-to-end cell first.")